<a href="https://colab.research.google.com/github/Fata-Haidar/244107020108-Pembelajaran-Mesin/blob/main/JS09/lab4_js09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Langkah 1 – Instal Library
Pasang library ANNOY, FAISS versi CPU, dan hnswlib. Jalankan satu kali pada sesi Google Colab atau lingkungan Python yang digunakan.

In [1]:
!pip install -q annoy faiss-cpu hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Langkah 2 – Import Library
Import NumPy untuk pengolahan data, `time` untuk mengukur performa, ANNOY, FAISS, dan hnswlib untuk membangun indeks.

In [2]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

print('Library berhasil diimpor.')

Library berhasil diimpor.


## Langkah 3 – Membuat Dataset
Modul menggunakan satu juta baris data dengan lima dimensi. Untuk perangkat dengan memori terbatas, gunakan 100.000 baris terlebih dahulu. Query dibuat sebagai satu vektor acak dengan dimensi yang sama.

In [3]:
# Build dataset with 1 million rows and 5 dimensions
n_data = 1_000_000  # gunakan 100_000 jika memori terbatas
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

print('Jumlah data:', n_data)
print('Dimensi:', dim)
print('Ukuran dataset (MB):', round(X.nbytes / (1024 ** 2), 2))
print('Query point:', query)
print('Jumlah tetangga:', k)

Jumlah data: 1000000
Dimensi: 5
Ukuran dataset (MB): 19.07
Query point: [[0.39642438 0.67123944 0.91809475 0.9920887  0.85126275]]
Jumlah tetangga: 10


## Langkah 4 – ANNOY
ANNOY membangun indeks berbasis beberapa tree. Pada percobaan ini digunakan metrik Euclidean dan 10 tree. Pengukuran `build_time` mencakup penambahan item ke indeks dan proses `build(10)`, sesuai alur modul. Pengukuran `query_time` hanya mencakup pencarian 10 tetangga.

In [4]:
# ANNOY
print('=== Annoy ===')
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.perf_counter()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
annoy_build_time = time.perf_counter() - start

start = time.perf_counter()
annoy_neighbors = ann_index.get_nns_by_vector(
    query[0], k, include_distances=True
)
annoy_query_time = time.perf_counter() - start

print('Build time:', annoy_build_time, 'detik')
print('Query time:', annoy_query_time, 'detik')
print('Neighbors:', annoy_neighbors[0][:5], '...')
print('Distances:', annoy_neighbors[1][:5], '...')

=== Annoy ===
Build time: 20.14258165399997 detik
Query time: 0.00030679299993607856 detik
Neighbors: [938362, 484047, 444455, 594631, 701660] ...
Distances: [0.06223752722144127, 0.06918369978666306, 0.07174362987279892, 0.07522597908973694, 0.07619021087884903] ...


## Langkah 5 – FAISS (IndexFlatL2)
`IndexFlatL2` merupakan pencarian exact dengan membandingkan query terhadap seluruh vektor. Proses `add(X)` diukur sebagai build time, sedangkan pemanggilan `search()` diukur sebagai query time.

**Catatan:** FAISS `IndexFlatL2` mengembalikan squared L2 distance (kuadrat jarak Euclidean).

In [5]:
# FAISS (Flat Index)
print('\n=== FAISS (IndexFlatL2) ===')
faiss_index = faiss.IndexFlatL2(dim)

start = time.perf_counter()
faiss_index.add(X)
faiss_build_time = time.perf_counter() - start

start = time.perf_counter()
faiss_distances, faiss_indices = faiss_index.search(query, k)
faiss_query_time = time.perf_counter() - start

print('Build time:', faiss_build_time, 'detik')
print('Query time:', faiss_query_time, 'detik')
print('Neighbors:', faiss_indices[0][:5], '...')
print('Squared L2 distances:', faiss_distances[0][:5], '...')


=== FAISS (IndexFlatL2) ===
Build time: 0.01613928200004011 detik
Query time: 0.009037730999921223 detik
Neighbors: [938362 484047 444455 594631 701660] ...
Squared L2 distances: [0.00387383 0.00478697 0.00514746 0.00565863 0.00580502] ...


## Langkah 6 – HNSW (hnswlib)
HNSW menggunakan struktur graph bertingkat. Parameter `M=16` mengatur keterhubungan graph, `ef_construction=200` memengaruhi kualitas konstruksi indeks, dan `ef=50` mengatur pencarian. Build time diukur dari inisialisasi indeks hingga seluruh data ditambahkan. Query time hanya mengukur `knn_query()`.

**Catatan:** `space='l2'` pada hnswlib menghasilkan squared L2 distance.

In [6]:
# HNSW (hnswlib)
print('\n=== HNSW (hnswlib) ===')
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.perf_counter()
hnsw_index.init_index(
    max_elements=n_data, ef_construction=200, M=16
)
hnsw_index.add_items(X)
hnsw_build_time = time.perf_counter() - start

hnsw_index.set_ef(50)

start = time.perf_counter()
hnsw_labels, hnsw_distances = hnsw_index.knn_query(query, k=k)
hnsw_query_time = time.perf_counter() - start

print('Build time:', hnsw_build_time, 'detik')
print('Query time:', hnsw_query_time, 'detik')
print('Neighbors:', hnsw_labels[0][:5], '...')
print('Squared L2 distances:', hnsw_distances[0][:5], '...')


=== HNSW (hnswlib) ===
Build time: 164.163484885 detik
Query time: 0.0003996389999656458 detik
Neighbors: [938362 484047 444455 594631 701660] ...
Squared L2 distances: [0.00387351 0.00478638 0.00514715 0.00565895 0.00580495] ...


## Langkah 7 – Perbandingan Performa
Gabungkan hasil ketiga metode ke dalam tabel. Build time dan query time harus dibaca secara terpisah: metode yang cepat saat mencari belum tentu paling cepat saat membangun indeks. Tampilkan lima indeks tetangga pertama seperti pada keluaran modul.

In [7]:
import pandas as pd

results = pd.DataFrame({
    'Metode': ['ANNOY', 'FAISS IndexFlatL2', 'HNSW'],
    'Build time (detik)': [annoy_build_time, faiss_build_time, hnsw_build_time],
    'Query time (detik)': [annoy_query_time, faiss_query_time, hnsw_query_time],
    '5 tetangga pertama': [
        annoy_neighbors[0][:5],
        faiss_indices[0][:5].tolist(),
        hnsw_labels[0][:5].tolist()
    ]
})

display(results)

,Metode,Build time (detik),Query time (detik),5 tetangga pertama
0,ANNOY,20.142582,0.000307,"[938362, 484047, 444455, 594631, 701660]"
1,FAISS IndexFlatL2,0.016139,0.009038,"[938362, 484047, 444455, 594631, 701660]"
2,HNSW,164.163485,0.000400,"[938362, 484047, 444455, 594631, 701660]"


## Kesimpulan
Praktikum ini membandingkan ANNOY, FAISS `IndexFlatL2`, dan HNSW pada dataset berdimensi lima. ANNOY dan HNSW menggunakan struktur indeks untuk mempercepat pencarian approximate, sedangkan FAISS `IndexFlatL2` melakukan pencarian exact. Pemilihan metode terbaik bergantung pada kebutuhan: waktu pembangunan indeks, kecepatan pencarian, kebutuhan ketepatan hasil, kapasitas memori, dan skala data.

### Hal yang perlu dilaporkan
- Jumlah data dan dimensi yang benar-benar digunakan.
- Spesifikasi perangkat/lingkungan eksekusi.
- Build time dan query time setiap metode.
- Lima indeks tetangga pertama dari masing-masing metode.
- Analisis perbedaan hasil dan kesimpulan berdasarkan data aktual.